# ISM 6564 — Assignment 2: a similarity-search engine over a help-centre corpus

**Week 2 — Representing Text as Vectors.** Read `README.md` in this folder for
the deadline, the deliverables, and the rubric before you start.

You are building the retrieval layer for the Northwind Outfitters help centre:
128 support articles, 28 real customer questions, and a relevance set that says
which articles actually answer each question. Your job is to represent the
articles as vectors, retrieve with several similarity metrics, **measure** which
combination retrieves best, diagnose *why*, improve it, and then recommend one
configuration and defend it.

**How this notebook is scaffolded.** Every cell you must complete is marked
`# TODO`. Cells that follow a TODO check your work and print a clear message if
you have not filled it in yet, so the notebook runs top to bottom from the
moment you open it. Do not delete the check cells — they are how the work is
marked. Do not change the given helper functions; if you think one is wrong, say
so in your Task 6 write-up rather than editing it silently.

Submit with **all cells run and outputs visible**, plus your AI Usage Report.

In [1]:
import random
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.spatial import distance
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=UserWarning)
print(f"torch device: {device}")

torch device: cpu


---

## The collection

Three files, loaded local-first with a URL fallback so this notebook works in a
clone of the repo and in Colab:

| File | Contents |
|---|---|
| `help_center.csv` | 128 articles: `doc_id`, `category`, `title`, `text` |
| `queries.csv` | 28 customer questions: `query_id`, `query_text` |
| `relevance.csv` | 93 human judgements: `query_id`, `doc_id`, `relevance` |

`relevance` is **graded**, not binary: **2** means the article answers the
question, **1** means it is related and useful but is not the answer, and any
article not listed for a query is treated as **0**. Graded judgements are why
Task 3 uses nDCG and not just precision.

The searchable text for each article is its **title plus its body**. Titles in a
help centre are short and dense with exactly the terms a customer types, so
throwing them away costs you real retrieval quality — but they are also only a
handful of tokens, so concatenating them rather than scoring them separately is
the simple choice. It is the choice you are given; you do not have to agree with
it, and Task 6 is the place to say so.

In [2]:
BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-02-vectors-similarity/data/")


def load(name: str) -> pd.DataFrame:
    """Local first (works from the assignment folder or the repo root), then URL."""
    for candidate in (Path("../data") / name,
                      Path("data") / name,
                      Path("week-02-vectors-similarity/data") / name):
        if candidate.exists():
            return pd.read_csv(candidate)
    return pd.read_csv(BASE_URL + name)


docs = load("help_center.csv")
queries = load("queries.csv")
relevance = load("relevance.csv")

# The searchable text: title + body.
docs["searchable"] = docs["title"] + ". " + docs["text"]

CORPUS = docs["searchable"].tolist()
DOC_IDS = docs["doc_id"].tolist()
DOC_INDEX = {d: i for i, d in enumerate(DOC_IDS)}
QUERY_IDS = queries["query_id"].tolist()
QUERY_TEXTS = queries["query_text"].tolist()

# QRELS[query_id][doc_id] -> 1 or 2. Anything absent is relevance 0.
QRELS = {}
for row in relevance.itertuples():
    QRELS.setdefault(row.query_id, {})[row.doc_id] = int(row.relevance)

assert len(CORPUS) == 128 and len(QUERY_IDS) == 28
assert set(QRELS) == set(QUERY_IDS)
assert all(d in DOC_INDEX for rel in QRELS.values() for d in rel)
print(f"{len(CORPUS)} documents | {len(QUERY_IDS)} queries | "
      f"{sum(len(r) for r in QRELS.values())} judgements")

128 documents | 28 queries | 93 judgements


In [3]:
word_counts = docs["searchable"].str.split().str.len()
print("document length in words")
print(word_counts.describe().round(1).to_string())
print()
print(docs["category"].value_counts().to_string())
print()
print("--- three example documents ---")
for i in (0, 103, 69):
    print(f"[{DOC_IDS[i]}] ({word_counts[i]} words) {CORPUS[i][:150]}...")
print()
print("--- five example queries with their judged articles ---")
for qid, qtext in list(zip(QUERY_IDS, QUERY_TEXTS))[:5]:
    judged = ", ".join(f"{d}({g})" for d, g in sorted(QRELS[qid].items()))
    print(f"{qid}  {qtext!r}\n      -> {judged}")

document length in words
count    128.0
mean      46.3
std       15.3
min       19.0
25%       36.0
50%       44.5
75%       53.0
max       99.0

category
returns-refunds      16
shipping-delivery    15
payments-billing     15
product-care         13
warranty-repairs     13
account-login        12
sizing-fit           12
gift-cards-promos    11
website-technical    11
order-tracking       10

--- three example documents ---
[NW0001] (60 words) Standard return window. Most items bought from Northwind Outfitters can be returned within sixty days of the delivery date for a full refund to the or...
[NW0104] (19 words) Gift cards never expire. Northwind gift cards carry no expiry date and no dormancy fee. Balances remain valid indefinitely....
[NW0070] (99 words) Boot sizing and width. Hiking boots are built on a last that runs about half a size small, so order half a size above your everyday shoe. Feet swell o...

--- five example queries with their judged articles ---
Q01  "how do I retur

---

# Task 1 — Build the vector space (12 points)

Before you can retrieve anything you have to decide what a document *is*, as a
vector. Build **four** representations of the same corpus and characterise each
one numerically. Do not just build them — describe them, because the numbers you
produce here explain the retrieval results in Task 3.

| Key | Configuration |
|---|---|
| `count` | `CountVectorizer()` — defaults |
| `tfidf` | `TfidfVectorizer()` — defaults |
| `tfidf_mindf2` | `TfidfVectorizer(min_df=2)` |
| `tfidf_bigram` | `TfidfVectorizer(ngram_range=(1, 2))` |

**Deliverable:** a DataFrame `task1_table` with one row per configuration and
the columns produced by the given `describe()` helper, followed by a markdown
cell of 4–6 sentences reading the table: what `min_df=2` removes and what that
costs, and what bigrams buy you for the vocabulary size they cost.

In [4]:
def describe(name: str, vectorizer, X) -> dict:
    """GIVEN — do not modify. Numerical description of a document-term matrix."""
    n_docs, n_terms = X.shape
    dense_mb = n_docs * n_terms * 8 / 1e6            # float64, if it were dense
    sparse_mb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
    return {
        "config": name,
        "vocab": n_terms,
        "stored_nonzeros": X.nnz,
        "density_%": round(100 * X.nnz / (n_docs * n_terms), 4),
        "nnz_per_doc": round(X.nnz / n_docs, 1),
        "dense_MB": round(dense_mb, 3),
        "sparse_MB": round(sparse_mb, 3),
        "max_weight": round(float(X.max()), 4),
    }

In [5]:
#  (Task 1): fit each of the four vectorizers on CORPUS.
# Store them so later tasks can reuse them:
#   VECTORIZERS[key] -> the fitted vectorizer
#   MATRICES[key]    -> the document-term matrix it produced (sparse, 128 x V)
VECTORIZERS = {}
MATRICES = {}

configs = {
    "count": CountVectorizer(),
    "tfidf": TfidfVectorizer(),
    "tfidf_mindf2": TfidfVectorizer(min_df=2),
    "tfidf_bigram": TfidfVectorizer(ngram_range=(1, 2)),
}

for name, vectorizer in configs.items():
    VECTORIZERS[name] = vectorizer
    MATRICES[name] = vectorizer.fit_transform(CORPUS)

# build the summary table by calling describe() for each configuration.
task1_table = pd.DataFrame(
    [describe(name, VECTORIZERS[name], MATRICES[name]) for name in configs]
)

In [6]:
EXPECTED_KEYS = ["count", "tfidf", "tfidf_mindf2", "tfidf_bigram"]

if task1_table is None or not MATRICES:
    print("Task 1 not done yet — fill in the TODO above, then re-run.")
else:
    missing = [k for k in EXPECTED_KEYS if k not in MATRICES]
    assert not missing, f"missing configurations: {missing}"
    for k in EXPECTED_KEYS:
        assert MATRICES[k].shape[0] == 128, f"{k}: expected 128 rows"
    print(pd.DataFrame(task1_table).set_index("config").to_string()
          if not isinstance(task1_table, pd.DataFrame)
          else task1_table.to_string())

         config  vocab  stored_nonzeros  density_%  nnz_per_doc  dense_MB  sparse_MB  max_weight
0         count   1593             4686     2.2981         36.6     1.631      0.057     10.0000
1         tfidf   1593             4686     2.2981         36.6     1.631      0.057      0.6863
2  tfidf_mindf2    658             3751     4.4536         29.3     0.674      0.046      0.7428
3  tfidf_bigram   6090            10228     1.3121         79.9     6.236      0.123      0.4714


### Task 1 write-up

min_df=2 drops every term that is in exactly one document, 935 of the 1,593 unigram terms, which cuts vocabulary from 1,593 to 658 and average nonzeros per document from 36.6 to 29.3. Density almost doubles (2.30% to 4.45%) because the removed terms were the sparsest ones, not because the matrix got more informative. The retrieval risk is that on a 128 document corpus a term appearing once is usually the most diagnostic word a customer will actually type (a product name, backordered, a specific SKU like term), so trimming it away costs recall on the queries where lexical overlap already needed help, and Task 3 confirms that tfidf_mindf2 scores lowest of the three TF-IDF variants (nDCG@10 = 0.5981 vs 0.6476 for plain tfidf). Bigrams are the opposite, vocabulary goes up 3.8× (1,593 to 6,090) but nnz_per_doc only grows 2.2× (36.6 to 79.9) and density actually down to 1.31%, because most bigrams are rare by construction, and it pays off, since tfidf_bigram is the Task 3 winner (nDCG@10 = 0.6746). All four configurations is under 4.5% density, which means these matrices has be stored and multiplied as sparse (CSR), never densified: tfidf_bigram would cost 6.236 MB dense vs 0.123 MB sparse,which will widen as the corpus grows.

---

# Task 2 — Retrieve with four similarity metrics (18 points)

Write **one** function, `score_matrix`, that scores every query against every
document under a named metric. Everything downstream — search, evaluation,
diagnosis — is built on it, so getting the contract exactly right matters:

```
score_matrix(Q, X, metric) -> ndarray of shape (n_queries, n_docs)
```

**Higher must always mean more similar.** Euclidean and Jaccard are natively
*distances*, so you have to convert them. Return the negated distance for
Euclidean (`-d`), and the similarity `1 - jaccard_distance` for Jaccard. If you
return a distance where a similarity is expected, every ranking in this notebook
silently inverts and your Task 3 numbers will be near zero — which is a real
failure mode, not a hypothetical one.

The four metrics:

| `metric` | Definition | Notes |
|---|---|---|
| `"cosine"` | $\dfrac{q\cdot d}{\lVert q\rVert\,\lVert d\rVert}$ | `sklearn.metrics.pairwise.cosine_similarity` |
| `"dot"` | $q \cdot d$ | no normalisation at all |
| `"euclidean"` | $-\lVert q - d\rVert_2$ | `euclidean_distances`, negated |
| `"jaccard"` | $\dfrac{\lvert q \cap d\rvert}{\lvert q \cup d\rvert}$ on **binary** presence | binarise with `> 0`, then `distance.cdist(..., "jaccard")` |

`Q` and `X` may arrive as scipy sparse matrices. `cosine_similarity` and
`euclidean_distances` accept sparse input; `distance.cdist` does not, so
densify with `.toarray()` inside the Jaccard branch only. At 128 × ~1,600 that
is a trivial amount of memory — but say why it would not be at 128 million.

**Deliverable:** a working `score_matrix` that passes the given self-test, the
printed side-by-side top-5 for `Q02` and `Q23` under all four metrics on the
`count` matrix, and a markdown cell explaining one specific disagreement you
can see in that output.

In [7]:
# (Task 2): implement the four metrics.
def score_matrix(Q, X, metric: str) -> np.ndarray:
    """(n_queries, n_docs) similarity scores. HIGHER = more similar, always."""
    if metric == "cosine":
        return cosine_similarity(Q, X)
    elif metric == "dot":
        S = Q @ X.T
        return S.toarray() if hasattr(S, "toarray") else np.asarray(S)
    elif metric == "euclidean":
        return -euclidean_distances(Q, X)
    elif metric == "jaccard":
        Q_bin = (Q.toarray() if hasattr(Q, "toarray") else np.asarray(Q)) > 0
        X_bin = (X.toarray() if hasattr(X, "toarray") else np.asarray(X)) > 0
        return 1 - distance.cdist(Q_bin, X_bin, "jaccard")
    else:
        raise ValueError(f"unknown metric: {metric}")


METRICS = ["cosine", "dot", "euclidean", "jaccard"]

In [8]:
# GIVEN — do not modify. A hand-checkable self-test for score_matrix.
X_toy = np.array([[1, 0, 1],
                  [0, 1, 1],
                  [2, 0, 2]], dtype=float)
Q_toy = np.array([[1, 0, 0]], dtype=float)

EXPECTED_TOY = {
    "cosine":    [0.70710678, 0.0, 0.70710678],
    "dot":       [1.0, 0.0, 2.0],
    "euclidean": [-1.0, -1.73205081, -2.23606798],
    "jaccard":   [0.5, 0.0, 0.5],
}

task2_ok = True
for m, expected in EXPECTED_TOY.items():
    try:
        got = np.asarray(score_matrix(Q_toy, X_toy, m)).ravel()
    except NotImplementedError:
        print(f"{m:<10} not implemented yet")
        task2_ok = False
        continue
    ok = got.shape == (3,) and np.allclose(got, expected)
    task2_ok &= bool(ok)
    print(f"{m:<10} {'ok ' if ok else 'WRONG'} got={np.round(got, 6).tolist()} "
          f"expected={np.round(expected, 6).tolist()}")

print()
print("Task 2 self-test passed." if task2_ok
      else "Task 2 self-test NOT passing yet — fix score_matrix before continuing.")

cosine     ok  got=[0.707107, 0.0, 0.707107] expected=[0.707107, 0.0, 0.707107]
dot        ok  got=[1.0, 0.0, 2.0] expected=[1.0, 0.0, 2.0]
euclidean  ok  got=[-1.0, -1.732051, -2.236068] expected=[-1.0, -1.732051, -2.236068]
jaccard    ok  got=[0.5, 0.0, 0.5] expected=[0.5, 0.0, 0.5]

Task 2 self-test passed.


In [9]:
# GIVEN — do not modify. Built on your score_matrix.
def rank_all(Q, X, metric: str) -> np.ndarray:
    """(n_queries, n_docs) array of document indices, best first."""
    S = np.asarray(score_matrix(Q, X, metric))
    return np.argsort(-S, axis=1, kind="stable")


def ranked_ids(Q, X, metric: str) -> dict:
    """query_id -> full list of doc_ids, best first."""
    order = rank_all(Q, X, metric)
    return {qid: [DOC_IDS[j] for j in order[i]] for i, qid in enumerate(QUERY_IDS)}


def top_k_table(qid: str, key: str, k: int = 5) -> pd.DataFrame:
    """Side-by-side top-k under all four metrics for one query."""
    vec, X = VECTORIZERS[key], MATRICES[key]
    Q = vec.transform(QUERY_TEXTS)
    i = QUERY_IDS.index(qid)
    cols = {}
    for m in METRICS:
        S = np.asarray(score_matrix(Q, X, m))[i]
        order = np.argsort(-S, kind="stable")[:k]
        cols[m] = [f"{DOC_IDS[j]} g{QRELS[qid].get(DOC_IDS[j], 0)} ({S[j]:.3f})"
                   for j in order]
    return pd.DataFrame(cols, index=[f"#{r}" for r in range(1, k + 1)])

In [10]:
# GIVEN — do not modify. "g2" = judged as answering the question, "g1" = related,
# "g0" = not judged relevant.
if not task2_ok or not MATRICES:
    print("Finish Tasks 1 and 2 first — then this cell shows the comparison.")
else:
    for qid in ("Q02", "Q23"):
        i = QUERY_IDS.index(qid)
        print(f"{qid}: {QUERY_TEXTS[i]!r}   (on the `count` matrix)")
        print(top_k_table(qid, "count").to_string())
        print()

Q02: 'does it cost anything to send an item back'   (on the `count` matrix)
               cosine                dot           euclidean            jaccard
#1  NW0027 g0 (0.240)  NW0027 g0 (8.000)  NW0104 g0 (-5.831)  NW0029 g0 (0.143)
#2  NW0029 g0 (0.216)  NW0009 g0 (7.000)  NW0029 g0 (-6.245)  NW0011 g0 (0.103)
#3  NW0010 g0 (0.202)  NW0015 g0 (7.000)  NW0037 g0 (-6.245)  NW0087 g0 (0.100)
#4  NW0011 g0 (0.192)  NW0041 g0 (7.000)  NW0053 g0 (-6.245)  NW0094 g0 (0.100)
#5  NW0096 g0 (0.189)  NW0010 g0 (6.000)  NW0064 g0 (-6.245)  NW0010 g0 (0.087)

Q23: 'check the balance left on a gift card'   (on the `count` matrix)
               cosine                 dot           euclidean            jaccard
#1  NW0103 g2 (0.591)  NW0103 g2 (15.000)  NW0104 g1 (-5.292)  NW0103 g2 (0.192)
#2  NW0108 g0 (0.546)  NW0041 g0 (14.000)  NW0053 g0 (-5.745)  NW0105 g1 (0.154)
#3  NW0049 g0 (0.442)  NW0108 g0 (14.000)  NW0101 g0 (-5.916)  NW0108 g0 (0.133)
#4  NW0006 g0 (0.431)  NW0079 g0 (13.000)  NW010

### Task 2 write-up

On Q23 (“check the balance left on a gift card”), cosine, dot, and Jaccard all correctly place NW0103 (g2, the article that answers the question) at rank 1. Euclidean does not, it puts NW0104 (g1, only “related”) at #1 and doesn't surface NW0103 in its top 5 at all. The mechanism is the short document bias built into Euclidean distance, NW0104 is one of the shortest documents in the corpus (19 words, “Gift cards never expire”), so its raw count vector has a little total magnitude, and a query vector can't be far from a document that barely has any nonzero counts to disagree with. NW0103 is longer and genuinely on topic, but it's extra term mass (including terms the query doesn't ask about) inflates the raw Euclidean distance even though it shares more of the right terms with the query. Cosine and dot avoid this because they only ever add positive contributions from shared terms instead of accumulating squared differences over every dimension, so a short and sparse document like NW0104 can't win purely by having little to be wrong about.

---

# Task 3 — Measure retrieval quality (22 points)

Everything so far has been eyeballing. Now measure. You will implement three
retrieval metrics; `ndcg_at_k` is given because graded-relevance nDCG has enough
fiddly detail to be worth reading rather than re-deriving.

| Metric | Definition |
|---|---|
| `precision_at_k(ranked, qrel, k=5)` | fraction of the top `k` with relevance > 0 |
| `recall_at_k(ranked, qrel, k=10)` | of all judged-relevant documents for this query, the fraction appearing in the top `k` |
| `reciprocal_rank(ranked, qrel)` | `1 / rank` of the **first** relevant document; `0.0` if none appears |
| `ndcg_at_k(ranked, qrel, k=10)` | **given** — gain $2^{g}-1$, discount $\log_2(\text{rank}+1)$, divided by the ideal DCG |

**A ceiling to keep in mind:** most queries have only 3 judged-relevant
articles, so mean P@5 cannot exceed **0.664** on this collection. Do not report
P@5 = 0.34 as though 1.0 were reachable. State the ceiling when you report the
number.

Then produce `task3_table`: mean P@5, R@10, MRR, and nDCG@10 over all 28
queries, for every combination of the four representations from Task 1 and the
four metrics from Task 2 — sixteen rows.

**Deliverable:** the four metric functions passing the given unit test, the
sixteen-row `task3_table`, and a markdown cell of 5–8 sentences. Two things in
that table are surprising. Find both and explain them; one of them has a
provable explanation you rehearsed in the practice notebook.

In [11]:
def ndcg_at_k(ranked, qrel: dict, k: int = 10) -> float:
    """GIVEN — do not modify. Normalised discounted cumulative gain, graded relevance.

    Gain for a document of relevance g is 2**g - 1, so a grade-2 article is
    worth 3 and a grade-1 article is worth 1: answering the question is three
    times as valuable as being merely related. The discount 1/log2(rank+1)
    means rank 1 counts fully, rank 2 counts 0.63, rank 10 counts 0.29.
    Dividing by the ideal DCG puts every query on a 0-1 scale regardless of how
    many relevant documents it has.
    """
    gains = [2 ** qrel.get(d, 0) - 1 for d in ranked[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = [2 ** g - 1 for g in sorted(qrel.values(), reverse=True)[:k]]
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

In [12]:
# (Task 3): implement the three remaining metrics.
# `ranked` is a list of doc_ids best-first; `qrel` is {doc_id: 1 or 2}.
# Anything not in `qrel` has relevance 0.

def precision_at_k(ranked, qrel: dict, k: int = 5) -> float:
    top_k = ranked[:k]
    relevant_in_top_k = sum(1 for d in top_k if qrel.get(d, 0) > 0)
    return relevant_in_top_k / k


def recall_at_k(ranked, qrel: dict, k: int = 10) -> float:
    relevant_docs = set(qrel.keys())
    if not relevant_docs:
        return 0.0
    found = relevant_docs & set(ranked[:k])
    return len(found) / len(relevant_docs)


def reciprocal_rank(ranked, qrel: dict) -> float:
    for rank, d in enumerate(ranked, start=1):
        if qrel.get(d, 0) > 0:
            return 1.0 / rank
    return 0.0

In [13]:
# GIVEN — do not modify. Hand-checkable unit test.
#   ranked  = A B C D E F G H I J
#   qrel    = {B: 2, D: 1, K: 2}   (K is never retrieved)
#   P@5     = 2 relevant in the top 5                     = 0.4
#   R@10    = 2 of the 3 judged-relevant documents found  = 0.666667
#   RR      = first relevant is B at rank 2               = 0.5
#   nDCG@10 = 2.3234658 / 5.3927893                       = 0.4308468
TOY_RANKED = list("ABCDEFGHIJ")
TOY_QREL = {"B": 2, "D": 1, "K": 2}
TOY_EXPECTED = {
    "precision_at_k": (precision_at_k, 0.4),
    "recall_at_k": (recall_at_k, 2 / 3),
    "reciprocal_rank": (reciprocal_rank, 0.5),
    "ndcg_at_k": (ndcg_at_k, 0.43084676706),
}

task3_ok = True
for name, (fn, want) in TOY_EXPECTED.items():
    got = fn(TOY_RANKED, TOY_QREL)
    if got is None:
        print(f"{name:<17} not implemented yet")
        task3_ok = False
        continue
    ok = np.isclose(got, want)
    task3_ok &= bool(ok)
    print(f"{name:<17} {'ok ' if ok else 'WRONG'} got={got:.9f} expected={want:.9f}")

print()
print("Task 3 metric functions passing." if task3_ok
      else "Task 3 metrics NOT passing yet — fix them before running the grid.")

precision_at_k    ok  got=0.400000000 expected=0.400000000
recall_at_k       ok  got=0.666666667 expected=0.666666667
reciprocal_rank   ok  got=0.500000000 expected=0.500000000
ndcg_at_k         ok  got=0.430846767 expected=0.430846767

Task 3 metric functions passing.


In [14]:
# GIVEN — do not modify. Runs one (representation, metric) combination end to end.
def evaluate(key: str, metric: str) -> dict:
    """Mean P@5, R@10, MRR, nDCG@10 over all 28 queries."""
    vec, X = VECTORIZERS[key], MATRICES[key]
    Q = vec.transform(QUERY_TEXTS)
    rankings = ranked_ids(Q, X, metric)
    rows = [
        (precision_at_k(rankings[qid], QRELS[qid], 5),
         recall_at_k(rankings[qid], QRELS[qid], 10),
         reciprocal_rank(rankings[qid], QRELS[qid]),
         ndcg_at_k(rankings[qid], QRELS[qid], 10))
        for qid in QUERY_IDS
    ]
    means = np.mean(np.array(rows, dtype=float), axis=0)
    return {"representation": key, "metric": metric,
            "P@5": round(means[0], 4), "R@10": round(means[1], 4),
            "MRR": round(means[2], 4), "nDCG@10": round(means[3], 4)}


def evaluate_custom(vec, X, metric: str, label: str) -> dict:
    """Same, for a representation you build yourself in Task 5."""
    Q = vec.transform(QUERY_TEXTS) if hasattr(vec, "transform") else vec
    rankings = ranked_ids(Q, X, metric)
    rows = [
        (precision_at_k(rankings[qid], QRELS[qid], 5),
         recall_at_k(rankings[qid], QRELS[qid], 10),
         reciprocal_rank(rankings[qid], QRELS[qid]),
         ndcg_at_k(rankings[qid], QRELS[qid], 10))
        for qid in QUERY_IDS
    ]
    means = np.mean(np.array(rows, dtype=float), axis=0)
    return {"representation": label, "metric": metric,
            "P@5": round(means[0], 4), "R@10": round(means[1], 4),
            "MRR": round(means[2], 4), "nDCG@10": round(means[3], 4)}


P_AT_5_CEILING = float(np.mean([min(len(QRELS[q]), 5) / 5 for q in QUERY_IDS]))
print(f"mean P@5 ceiling on this collection: {P_AT_5_CEILING:.4f}")

mean P@5 ceiling on this collection: 0.6643


In [15]:
# (Task 3): build the 16-row results table, one row per
# (representation, metric) pair, using evaluate(). Sort it so the best
# nDCG@10 is at the top.
task3_table = pd.DataFrame(
    [evaluate(key, metric) for key in EXPECTED_KEYS for metric in METRICS]
).sort_values("nDCG@10", ascending=False).reset_index(drop=True)

In [16]:
if task3_table is None or not task2_ok or not task3_ok:
    print("Task 3 not done yet — finish Tasks 1-3, then re-run.")
else:
    t3 = pd.DataFrame(task3_table) if not isinstance(task3_table, pd.DataFrame) else task3_table
    assert len(t3) == 16, f"expected 16 rows, got {len(t3)}"
    print(t3.to_string(index=False))
    print()
    print(f"(mean P@5 cannot exceed {P_AT_5_CEILING:.4f} on this collection)")

representation    metric    P@5   R@10    MRR  nDCG@10
  tfidf_bigram euclidean 0.3286 0.5869 0.8315   0.6746
  tfidf_bigram    cosine 0.3286 0.5869 0.8315   0.6746
  tfidf_bigram       dot 0.3286 0.5869 0.8315   0.6746
         tfidf       dot 0.3214 0.5661 0.7858   0.6476
         tfidf    cosine 0.3214 0.5661 0.7858   0.6476
         tfidf euclidean 0.3214 0.5661 0.7858   0.6476
  tfidf_mindf2    cosine 0.3000 0.5750 0.7237   0.5981
  tfidf_mindf2       dot 0.3000 0.5750 0.7237   0.5981
  tfidf_mindf2 euclidean 0.3000 0.5750 0.7237   0.5981
         count       dot 0.2286 0.4786 0.5939   0.4927
         count    cosine 0.2214 0.4970 0.5833   0.4899
  tfidf_bigram   jaccard 0.2571 0.4756 0.5484   0.4743
         count   jaccard 0.2214 0.4845 0.4700   0.4160
         tfidf   jaccard 0.2214 0.4845 0.4700   0.4160
  tfidf_mindf2   jaccard 0.2214 0.4440 0.4694   0.3868
         count euclidean 0.0143 0.0357 0.0746   0.0153

(mean P@5 cannot exceed 0.6643 on this collection)


### Task 3 write-up

tfidf_bigram and any of cosine, dot, or euclidean wins with nDCG@10 = 0.6746, about 4 points above plain tfidf (0.6476) and about 8 points above tfidf_mindf2 (0.5981), it beats every jaccard (best jaccard is tfidf_bigram+jaccard at 0.4743). Two things in the table are surprising, first, for every TF-IDF representation, cosine, dot, and euclidean produce identical P@5/R@10/MRR/nDCG@10, not just close, identical. This is pobably because TfidfVectorizer L2 normalises every document vector by default, so all document vectors have norm 1, for unit vectors, dot product is cosine similarity, and squared Euclidean distance collapses to 2 - 2·(dot product), a decreasing function of the dot product, so all three have the exact same ranking no matter the query normalisation. Second, count+euclidean is not good (nDCG@10 = 0.0153, which is barely above chance), which is worse than count+cosine (0.4899) on the same matrix. Raw counts are not normalised, so this is the document length bias from Task 4 in, unnormalised Euclidean distance on raw counts is basically unusable for retrieval. Whether tfidf_bigram would still be the choice at 100× the corpus size is a trade off and not a clear yes because it's ranking advantage over plain tfidf here is just ok (about 4% relative nDCG) and its vocabulary cost is 3.8× larger and bigram vocabularies grow faster than unigram vocabularies when more documents are added, so the storage/compute premium would grow even though the accuracy gap may not.

---

# Task 4 — Diagnose the collection (16 points)

Two specific pathologies, measured rather than asserted.

### 4a — Document-length bias

Document length in this corpus ranges from 19 to 99 words (title plus body). On unnormalised
counts, a longer document has more of every term and therefore a higher raw dot
product with almost any query. Quantify it:

1. Compute each document's total token count from the `count` matrix.
2. For `metric="dot"` and `metric="cosine"` on the `count` matrix, compute each
   document's **mean score across all 28 queries**, and then the Pearson
   correlation between that mean score and the document's token count.
3. Report the mean token length of the rank-1 document under each metric, and
   compare both against the corpus mean.

### 4b — Near-duplicate audit

Help centres accumulate revisions. A policy page and its updated successor both
sit in the index, both match the same query, and together they burn two of your
five top slots to say one thing.

Find them with Jaccard over word shingles — the same machinery as the practice
notebook, over all $\binom{128}{2} = 8{,}128$ pairs:

1. Build 5-word shingle sets for all 128 documents.
2. Compute pairwise Jaccard.
3. Look at the **distribution** before picking a threshold, then report every
   pair above it with its titles.

**Deliverable:** the numbers from 4a, the near-duplicate pairs from 4b, and a
markdown cell stating what each finding means for the recommendation you make in
Task 6. There are exactly four near-duplicate pairs planted in this corpus and
they are cleanly separated from everything else — if your threshold produces 40
pairs or 0 pairs, your threshold is wrong, not the corpus.

In [17]:
# (Task 4a): quantify the document-length bias on the `count` matrix.
# Produce:
#   doc_tokens         (128,) total token count per document
#   length_corr        {"dot": float, "cosine": float} Pearson r between a
#                      document's mean score across the 28 queries and its length
#   rank1_mean_length  {"dot": float, "cosine": float} mean token length of the
#                      rank-1 document, averaged over the 28 queries
X_count = MATRICES["count"]
Q_count = VECTORIZERS["count"].transform(QUERY_TEXTS)

doc_tokens = np.asarray(X_count.sum(axis=1)).ravel()

length_corr = {}
rank1_mean_length = {}
for m in ("dot", "cosine"):
    S = np.asarray(score_matrix(Q_count, X_count, m))
    mean_score_per_doc = S.mean(axis=0)
    length_corr[m] = float(np.corrcoef(mean_score_per_doc, doc_tokens)[0, 1])

    order = rank_all(Q_count, X_count, m)
    rank1_idx = order[:, 0]
    rank1_mean_length[m] = float(doc_tokens[rank1_idx].mean())

In [18]:
if doc_tokens is None or length_corr is None or rank1_mean_length is None:
    print("Task 4a not done yet — fill in the TODO above, then re-run.")
else:
    corpus_mean = float(np.asarray(doc_tokens).mean())
    print(f"corpus mean document length: {corpus_mean:.1f} tokens")
    print()
    print(f"{'metric':<10}{'corr(mean score, length)':>26}{'mean length of rank-1 doc':>28}")
    for m in ("dot", "cosine"):
        print(f"{m:<10}{length_corr[m]:>26.3f}{rank1_mean_length[m]:>28.1f}")

corpus mean document length: 45.2 tokens

metric      corr(mean score, length)   mean length of rank-1 doc
dot                            0.725                        61.7
cosine                         0.471                        48.5


In [19]:
# (Task 4b): near-duplicate audit.
# Produce:
#   shingle_sets    list of 128 sets of 5-word shingles
#   pair_scores     list of (jaccard, doc_id_a, doc_id_b) for all 8,128 pairs
#   dup_threshold   the cut-off you chose, justified by the distribution
#   near_dupes      DataFrame of the pairs above dup_threshold, with both titles

def make_shingles(text: str, n: int = 5) -> set:
    words = text.lower().split()
    return set(tuple(words[i:i + n]) for i in range(len(words) - n + 1))


shingle_sets = [make_shingles(doc, 5) for doc in CORPUS]

pair_scores = []
for i, j in combinations(range(len(DOC_IDS)), 2):
    a, b = shingle_sets[i], shingle_sets[j]
    union = a | b
    jaccard = len(a & b) / len(union) if union else 0.0
    pair_scores.append((jaccard, DOC_IDS[i], DOC_IDS[j]))

# The distribution has a huge gap: the top 4 pairs score 0.65-0.83, then the
# next-highest pair drops to 0.03 — over 20x smaller. Any threshold between
# ~0.05 and ~0.6 cleanly separates the planted duplicates from the rest.
dup_threshold = 0.3

title_by_id = dict(zip(docs["doc_id"], docs["title"]))
near_dupes = pd.DataFrame([
    {"jaccard": round(jac, 4), "doc_id_a": a, "doc_id_b": b,
     "title_a": title_by_id[a], "title_b": title_by_id[b]}
    for jac, a, b in pair_scores if jac > dup_threshold
]).sort_values("jaccard", ascending=False).reset_index(drop=True)

In [20]:
if pair_scores is None or near_dupes is None:
    print("Task 4b not done yet — fill in the TODO above, then re-run.")
else:
    scores = np.array([p[0] for p in pair_scores], dtype=float)
    assert len(scores) == 8128, f"expected 8128 pairs, got {len(scores)}"
    print(f"pairwise 5-shingle Jaccard over {len(scores)} pairs")
    print(f"  max      {scores.max():.4f}")
    print(f"  99.9th % {np.percentile(scores, 99.9):.4f}")
    print(f"  99th %   {np.percentile(scores, 99):.4f}")
    print(f"  median   {np.median(scores):.4f}")
    print(f"  exactly zero: {(scores == 0).sum()} pairs "
          f"({100 * (scores == 0).mean():.1f}%)")
    print()
    print(f"threshold chosen: {dup_threshold}")
    nd = pd.DataFrame(near_dupes) if not isinstance(near_dupes, pd.DataFrame) else near_dupes
    print(nd.to_string(index=False))

pairwise 5-shingle Jaccard over 8128 pairs
  max      0.8250
  99.9th % 0.0095
  99th %   0.0000
  median   0.0000
  exactly zero: 8116 pairs (99.9%)

threshold chosen: 0.3
 jaccard doc_id_a doc_id_b                  title_a                                title_b
  0.8250   NW0040   NW0127 Accepted payment methods              Payment options we accept
  0.7385   NW0001   NW0125   Standard return window          Return window (2026 revision)
  0.6667   NW0016   NW0126  Standard delivery times  Standard delivery estimates (updated)
  0.6538   NW0091   NW0128 What the warranty covers Scope of the limited lifetime warranty


### Task 4 write-up

The length correlation numbers confirm the bias is real on raw counts, dot product's mean score per document correlates with document length at r = 0.725, and the average rank 1 document under dot is 61.7 tokens, 36% longer than the corpus mean of 45.2, so dot is rewarding length instead of relevance. TF-IDF's default L2 normalisation mitigates the problem (cosine's correlation drops to r = 0.471 and it's rank 1 mean length drops to 48.5, only 7% above the corpus mean) but it does not make it disappear, normalising away raw magnitude still leaves longer documents with broad vocabulary so more chances to overlap with any given query, so a residual length advantage survives. In production I would deduplicate at index time instead of collapsing at query time or leave the duplicates in place. Leaving them with costs search quality directly, a payments query that matches both NW0040 and NW0127 (jaccard 0.825, both about accepted payment methods) burns two of five top slots to say the same thing once. Collapsing at query time requires recomputing almost duplicate clusters against every result set, which adds latency and complexity to every query for a problem that is honestly probably a one time issue. Index time dedup is a single audit run per content update (keep the newer revision, so “Return window (2026 revision)” over the original, and stop the older doc_id), which is cheap here (4 clean pairs out of 8,128) and would scale the same way at any corpus size.

---

# Task 5 — Improve it, twice (18 points)

Pick **two** changes, implement them, and re-evaluate with `evaluate_custom()`.
Choose from this menu or propose your own:

| Change | What it targets |
|---|---|
| `sublinear_tf=True` | a term appearing 8 times is not 8× as informative as one appearing once |
| `stop_words="english"` | function words dominating the dot product |
| `min_df` / `max_df` tuning | vocabulary noise vs vocabulary coverage |
| `ngram_range=(1, 2)` | short phrases the unigram model cannot see |
| `analyzer="char_wb", ngram_range=(3, 5)` | morphology and misspellings — `delete`/`deleting`, `reproof`/`reproofing` |
| `TruncatedSVD` (LSA) to 50–150 dimensions | vocabulary mismatch between how customers write and how articles are written |
| deduplicating the Task 4b near-duplicates before indexing | wasted top-k slots |

**Rules.** Change one thing at a time so you can attribute the effect. Keep the
metric fixed at your Task 3 winner unless the change makes another metric
mandatory (SVD produces a dense matrix with negative entries — think about what
that does to Jaccard, and to Euclidean). Report all four metrics, not just the
one that improved.

**A warning that is worth marks.** You are tuning against the same 28 queries
you are scoring on. With no held-out set, "improvement" here means "fits this
relevance set better", which is not the same as "will retrieve better next
month". Your Task 6 memo must acknowledge this and say what you would do about
it.

**Deliverable:** `task5_table` showing the baseline and both variants across all
four metrics, plus a markdown cell of 4–6 sentences per change: what you
predicted, what happened, and the mechanism. A change that made things *worse*
and is honestly diagnosed earns full marks; an unexplained improvement does
not.

In [21]:
# (Task 5): implement your two changes.
# Build each variant, evaluate it with evaluate_custom(vec, X, metric, label),
# and collect the baseline + both variants into task5_table.
#
# Baseline: the Task 3 winner, tfidf_bigram + cosine (nDCG@10 = 0.6746).
baseline = evaluate("tfidf_bigram", "cosine")
baseline["representation"] = "tfidf_bigram (baseline)"

# Change 1: sublinear_tf=True on top of the winning representation — replaces
# raw term frequency with 1 + log(tf), so a term appearing 8 times is not
# treated as 8x as informative as one appearing once.
vec_sublinear = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_sublinear = vec_sublinear.fit_transform(CORPUS)
variant_sublinear = evaluate_custom(vec_sublinear, X_sublinear, "cosine",
                                     "tfidf_bigram + sublinear_tf")

# Change 2: stop_words="english" on top of the winning representation —
# removes common function words (the, is, to, ...) that add noise to the
# similarity score without carrying topical meaning.
vec_stopwords = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
X_stopwords = vec_stopwords.fit_transform(CORPUS)
variant_stopwords = evaluate_custom(vec_stopwords, X_stopwords, "cosine",
                                     "tfidf_bigram + stop_words")

task5_table = pd.DataFrame([baseline, variant_sublinear, variant_stopwords])

In [22]:
if task5_table is None or not task3_ok:
    print("Task 5 not done yet — fill in the TODO above, then re-run.")
else:
    t5 = pd.DataFrame(task5_table) if not isinstance(task5_table, pd.DataFrame) else task5_table
    assert len(t5) >= 3, "expected a baseline row plus two variants"
    print(t5.to_string(index=False))
    print()
    base = t5.iloc[0]
    for _, row in t5.iloc[1:].iterrows():
        deltas = " ".join(f"{c} {row[c] - base[c]:+.4f}"
                          for c in ("P@5", "R@10", "MRR", "nDCG@10"))
        print(f"{row['representation']:<28} vs baseline: {deltas}")

             representation metric    P@5   R@10    MRR  nDCG@10
    tfidf_bigram (baseline) cosine 0.3286 0.5869 0.8315   0.6746
tfidf_bigram + sublinear_tf cosine 0.3357 0.5923 0.8249   0.6887
  tfidf_bigram + stop_words cosine 0.3357 0.6798 0.7940   0.6899

tfidf_bigram + sublinear_tf  vs baseline: P@5 +0.0071 R@10 +0.0054 MRR -0.0066 nDCG@10 +0.0141
tfidf_bigram + stop_words    vs baseline: P@5 +0.0071 R@10 +0.0929 MRR -0.0375 nDCG@10 +0.0153


### Task 5 write-up

sublinear_tf=True: I expected an ok gain, dampening raw term frequency with 1 + log(tf) should stop a document that happens to repeat a query term many times from dominating purely on repetition. That's what happened, small, broad-based gains in P@5 (+0.0071), R@10 (+0.0054), and nDCG@10 (+0.0141), but MRR fell slightly (-0.0066). The likely mechanism would be a document that used to rank #1 on the strength of one heavily repeated key term loses some of that edge once repetition is log-dampened, letting a close competitor tie or overtake it on at least one query, which costs reciprocal rank even as the overall ranking improves. The effect is just ok overall because these documents are short (about 45 words), leaving little room for term repetition to matter in the first place.

stop_words="english": I expected the bigger gain of the two, since removing function words should prune noisy bigrams like "the return" that carry no topical signal. That's basically what happened, especially on recall: R@10 jumped +0.0929 and nDCG@10 jumped +0.0153 (matching sublinear_tf), but MRR dropped -0.0375, the largest MRR cost of the two changes. The mechanism, removing stopwords shrinks the vocabulary and increases term overlap between documents (function word bigrams that used to distinguish two similarl -worded articles are gone), which pulls more borderline relevant documents into the top 10 (helping recall and nDCG) but can blur the single sharpest match at rank 1 (hurting MRR). For a help centre search box where users scan a short results list rather than trust only the first hit, I would weight R@10/nDCG@10 over MRR and prefer the stop words variant, but if the box only ever surfaces one answer, the MRR cost would argue for sublinear_tf instead.

---

# Task 6 — Recommendation (14 points)

Write a memo of **300–400 words** to the (fictional) Northwind support-platform
team, who have to ship one configuration next sprint. This is the task that is
marked on judgement rather than on code, and the one where an unsupported
assertion costs you most.

Your memo must contain all six of the following:

1. **The recommendation.** One representation, one metric, stated in a single
   sentence someone could implement from.
2. **The evidence.** The specific numbers from Tasks 3 and 5 that support it,
   including the one that argues *against* your choice — there always is one.
3. **What it costs.** Index size and query-time cost, using the figures from
   Task 1. Say what happens at 128,000 documents rather than 128.
4. **What it fails at.** Name at least two query types this configuration will
   handle badly, using specific query IDs from this collection as evidence, not
   speculation.
5. **The evaluation caveat.** 28 queries, 93 judgements, one judge, and you
   tuned on the same set you scored on. State plainly what that means about how
   much to trust the difference between your top two configurations.
6. **What would change your mind.** The specific measurement or observation that
   would make you switch, and how you would gather it.

Do not recommend "use embeddings instead". Dense retrieval is material this
course reaches later, and you have not measured it here; recommending an
unmeasured system is exactly the habit this task exists to break. You may name
it as future work under point 6, together with the evidence you would want
before committing to it.

### Memo

**To:** Northwind support-platform team
**From:** Hesham Rashid
**Re:** Retrieval configuration for the help-centre search box

**Recommendation.** Ship TfidfVectorizer(ngram_range=(1,2), stop_words="english") scored with cosine similarity.

**Evidence.** This configuration reached nDCG@10 = 0.6899 and R@10 = 0.6798, the best of everything tested, improving on the tfidf_bigram+cosine baseline (nDCG@10 = 0.6746, R@10 = 0.5869) and well ahead of plain tfidf (0.6476) and every Jaccard or raw count configuration. The evidence against it: MRR fell from 0.8315 (baseline) to 0.7940, the largest MRR cost of any change we tried, stop word removal buys recall and overall ranking quality at some expense to how reliably the single best answer lands in the very top spot.

**Cost.** At 128 documents the bigram vocabulary is 6,090 terms stored in 0.123 MB sparse (vs. 6.24 MB if ever densified, don't). At 128,000 documents, vocabulary growth is sublinear (Heaps law) but bigram counts still climb into the hundreds of thousands of unique terms, storage stays manageable in sparse form, but a brute force cosine scan against every document per query stops being "trivial" and becomes the bottleneck, so we would need an inverted index or approximate nearest-neighbour structure, not a linear scan.

**What it fails at.** 1) Vocabulary mismatch, Q02, “does it cost anything to send an item back,” returns zero relevant articles in the top 5 under any metric we tested on the count matrix, and lexical methods can't bridge “cost anything”/“send back” to how the articles actually say “refund”/“shipping fee.”. 2) Near-duplicate redundancy: a query that matches NW0040 and NW0127 (jaccard 0.825, both “accepted payment methods”) burns two of five top slots on one fact.

**Evaluation caveat.** We have 28 queries, 93 judgements, one judge, and we tuned the two changes in Task 5 on the same set we scored them on. The gap between our top two configurations (stop words nDCG@10 = 0.6899 vs. sublinear_tf 0.6887) is 0.0012 and even the gap over baseline (+0.0153) means “fits this judgement set slightly better,” and not “will retrieve better next month.”

**What would change my mind.** A held out query set would change my mind, which is new queries and judgements not used for tuning, that showed the stop-words variant's R@10/nDCG edge holding up would confirm the choice, if it favoured sublinear_tf or plain tfidf_bigram, then I'd switch. I'd also want a second judge on a sample of the existing 93 judgements to check agreement, since the whole ranking relies on one person's relevance calls. Longer term, dense embeddings are worth measuring once the course covers them, but not before we have numbers for them on this collection.

---

## Before you submit

- [X] **Kernel → Restart & Run All.** The notebook must run top to bottom in a
      fresh kernel. A notebook whose outputs came from cells run out of order is
      marked as it runs, not as it looks.
- [X] All outputs visible and saved in the `.ipynb` you upload.
- [X] Every self-test and check cell prints a passing result.
- [X] All six write-up markdown cells replaced with your own text — none of the
      italic placeholder prompts left in.
- [X] Every number you cite in a write-up appears in an output cell above it.
- [X] **AI Usage Report** submitted alongside the notebook. See
      [`templates/ai-usage-report.md`](../../templates/ai-usage-report.md) and
      §7 of the [syllabus](../../syllabus.md). It is required even if you used
      no AI, and it is worth up to −20 points if it is missing.